In [1]:
%run stdPackages.ipynb
os.chdir(d['py'])
from modelData import *
from gmsPython import nestingTree
from multOutIO import AdjustIO, CreateIOTargets, CreateIOTargets_HouseDemand

No clean-up of work-folder


# Create Model Data

## 0. Preliminaries

*Define main settings:*

In [2]:
t0 = 2019
T  = t0 + 100 # set finite time horizon
name = 'vGR' # Some global name that carries through all models/data to identify this version

*This creates a copy of the IO data from the ```IOdata``` project into the local data folder*

In [3]:
ioName = f'IO{t0}_WCGE_GR'
f_IOdata = os.path.join(d['project'], 'IOdata','data','processedData', ioName)
shutil.copy(f_IOdata, os.path.join(d['data'], ioName))

'C:\\Users\\sxj477\\Documents\\GitHub\\CGE_Generator\\projects\\WasteCGE\\data\\IO2019_WCGE_GR'

*Load data:*

In [4]:
db = GpyDB(os.path.join(d['data'], ioName), name = f'{name}{t0}_db', ws = d['work'])

*Load data on waste part as well:*

In [5]:
wasteData = DbFromExcel.simpleLoad(os.path.join(d['data'], 'WasteTemp.xlsx'))
wasteMaps = DbFromExcel.simpleLoad(os.path.join(d['data'], 'wasteDataMapping.xlsx'))

*For relevant variables, add yearly index to data (this is currently all of them)*

In [6]:
def add_t0(k):
    k.index = stdSort(pd.MultiIndex.from_frame(k.index.to_frame(index=False).assign(t = t0)))
[add_t0(db(k)) for k in db.getTypes(['var','par'])];

*Add various time subsets that we'll rely on:*

In [7]:
addTimeToDB(t0, T, db)

## 1. Adjust data for supply and demand

Load data:

In [8]:
def readSymbol(wasteData, sheet, columns = ['s','m','k','o']):
    df = pd.DataFrame(wasteData[sheet].values)
    return pd.DataFrame(df.iloc[1:,:].values, columns = df.iloc[0]).set_index(columns)[sheet[0]]
wd = {s: readSymbol(wasteData, s) for s in ('qd','qs','vd','vs')}

Check balances: For each combination of (m,k) the sum of demand equals sum of supply

In [9]:
def testBalance(d, s, error = 1e-6):
    return all(abs(d.groupby(['m','k']).sum()-s.groupby(['m','k']).sum())<error)
assert all([testBalance(wd[f'{k}d'], wd[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

Adhoc adjustment of industry names:

In [10]:
adhocNames = {'Export': 'F',' Import': 'F', 'Household': 'HH'}

In [11]:
for k in wd:
    wd[k].index = wd[k].index.set_levels(wd[k].index.levels[0].map(lambda x: adhocNames[x] if x in adhocNames else x), level = 0)

In [12]:
(wd['vs'].xs('010000')/1000000)

m           k  o
Biological  V  D    144.390767
            W  D       0.34203
Metal       W  D       0.01991
Paper       W  D      0.001015
Plastic     W  D       0.00537
Rubber      W  D      0.000958
Textile     V  D       0.00292
Name: v, dtype: object

In [13]:
(wd['vd'].xs('010000')/1000000)

m           k  o
Biological  V  D    80.560242
            W  D     0.293994
Plastic     V  D     0.000027
Name: v, dtype: object

### 1.1. Adhoc adjustments

#### A. Demand from "Nature"

We adjust materials data in two ways. First, the demand vector contains an industry refereced as "Nature". This is included for completeness, but does not represent any actual transaction (think of this as capturing emissions - i.e. residual output that is not kept in the economic system). As we currently only model transactions that are priced and valued, we remove the nature part of the demand vector and scale down all supplies appropriately to rebalance the system.

*Note: The following assumes that we only have "nature" in the demand vector, and not in the supply vector.*

Get shares of demand from nature:

In [14]:
nat = pd.Index(['Nature'], name = 's')
natDem = {k: adj.rc_pd(wd[k], nat).groupby(['m','k']).sum() for k in ('vd','qd')}
natDemShares = {k: natDem[k]/adj.rc_pd(wd[k], natDem[k]).groupby(['m','k']).sum() for k in ('vd','qd')}

Remove from demand vectors:

In [15]:
for k in ('vd','qd'):
    wd[k] = adj.rc_pd(wd[k], ('not',nat))
    wd[k].index = wd[k].index.remove_unused_levels()

Rebalance supply by adjusting downwards:

In [16]:
for k in ('q','v'):
    wd[f'{k}s'] = (adj.rc_pd(wd[f'{k}s'], natDemShares[f'{k}d']) * (1-natDemShares[f'{k}d'])).combine_first(wd[f'{k}s'])

Check balances again, just in case:

In [17]:
assert all([testBalance(wd[f'{k}d'], wd[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

#### B. Household supply

*Old comment:*
<!-- The household's supply are mostly materials that are collected and treated by the waste management industry. In practice, these transfers thus represent materials that the waste management industry can sell after incurring costs of transportation and treatment at waste plants. Because the governing IO matrix on firms does not contain *any* transfers from households to firms *except* for wages and interests on investments, the only consistent treatment of this part of the data is to add it directly to the waste industry and remove the household from the supply vector.

We try to still capture the basics of the waste collection/treatment flows as follows:
* Assume that waste collection costs are roughly proportional to the amount of waste they collect from households.
* Assume that households' waste generation is roughly proportional to the total consumption.
* In this case, modeling the households' demand for waste collection services as proportional to total consumption (leontief-like utility), more consumption -> more waste generation -> more demand for collection services. In the waste treatment sector, assuming that outputs of other materials are close to proportional to their "industry specific service" means that more waste treatment also results in more recycled materials.


Thus, the loop from higher consumption -> more waste generation -> more waste collection/treatment -> more output of recycled materials is present in the model. Beyond this, the technological options for *increasing* recycling is not accurately captured in a model with simply CES/CET functions. For this, we need additional data on the waste treatment sector. -->

For now, remove household supply and rebalance demands.

Get shares of supply from household:

In [18]:
hh = pd.Index(['HH'], name = 's')
hhSup = {k: adj.rc_pd(wd[k], hh).groupby(['m','k']).sum() for k in ('vs','qs')}
hhSupShares = {k: hhSup[k]/adj.rc_pd(wd[k], hhSup[k]).groupby(['m','k']).sum() for k in ('vs','qs')}

Remove from supply vectors:

In [19]:
for k in ('vs','qs'):
    wd[k] = adj.rc_pd(wd[k], ('not',hh))
    wd[k].index = wd[k].index.remove_unused_levels()

Rebalance demands:

In [20]:
for k in ('q','v'):
    wd[f'{k}d'] = (adj.rc_pd(wd[f'{k}d'], hhSupShares[f'{k}s']) * (1-hhSupShares[f'{k}s'])).combine_first(wd[f'{k}d'])

Check balances again, just in case:

In [21]:
assert all([testBalance(wd[f'{k}d'], wd[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

Rebalance to make sure that there are no small numerical errors:

In [22]:
[wd.__setitem__(f'{k}s', wd[f'{k}s'] * wd[f'{k}d'].groupby(['m','k']).sum()/wd[f'{k}s'].groupby(['m','k']).sum()) for k in ('q','v')];

Scale values to match database scale:

In [23]:
[wd.__setitem__(k, wd[k]*1000/db._scale) for k in ('vs','vd')];

### 1.2. Adjustments to materials data

#### A. Map to GR sectors

Map the $s$ set to the one applied in the model. To do so, we start by mapping the 121 sectors to the 146 sectors (one-to-many mapping). When needed, we use the relative sizes of the 146 sectors. Then, we go from the 146 to the roughly 50 sectors in GreenREFORM. This is also a many-to-one mapping - which is simple aggregation.

Database with 146 sectors:

In [24]:
db146 = GpyDB(os.path.join(d['project'], 'IOdata','data','processedData', f'IO{t0}_s146_dur7'), ws = db.ws)

Mappings from 121 to 146 and sGR:

In [25]:
df = pd.DataFrame(wasteMaps['s'].values).ffill()
df = pd.DataFrame(df.iloc[1:].values, columns = df.iloc[0]).astype(str)
df.loc[:,'sGR'] = df.loc[:,'sGR'].replace('35011','Energy') # local name for the energy industry
map121_146 = pd.MultiIndex.from_frame(df[['s121','s146']])
map146_sGR = pd.MultiIndex.from_frame(df[['s146','sGR']])

When mapping from 121 to 146 sectors, use relative size of the sectors:

In [26]:
sectorValue = adjMultiIndex.applyMult(adj.rc_pd(db146('vD').groupby('s').sum()+db146('TotalTax'), db146('s_p')).rename_axis('s146'),
                                      map121_146)
weights = sectorValue / (sectorValue.groupby('s121').sum())

Apply to get to 146 sectors:

In [27]:
wd146 = {}
for k in wd:
    wd146[k] = (adj.rc_pd(wd[k].rename_axis(index = {'s':'s121'}), weights) * weights).dropna().droplevel('s121')

Apply mapping from 146 to sGR:

In [28]:
wdGR = {}
for k in wd:
    wdGR[k] = pyDatabases.pdGb(adjMultiIndex.applyMult(wd146[k], map146_sGR), 's146').sum().rename_axis(index={'sGR':'s'}).reorder_levels(wd[k].index.names)

Combine, rename, clean up:

In [29]:
for k in wd:
    wd[k] = wdGR[k].combine_first(adj.rc_pd(wd[k], ('not', weights.index.levels[0].rename('s'))))

#### B. Adjustment to IO data from non-IO goods

Specify IO inputs to use in rebalancing:

In [30]:
sd = db('s_p').union(db('s_f')).union(db('s_HH')) # sectors used to rebalance demand
vD0 = adj.rc_pd(db('vD').xs(t0), ('and', [db('n_p').union(db('n_F')), sd]))

Create demand vector:

In [30]:
vDn = wd['vd'].rename('vd').reset_index()
vDn['n'] = vDn['m']+'_'+vDn['k']
vDn = vDn.set_index(['s','n'])['vd']

Supply vector:

In [31]:
vSn = wd['vs'].rename('vs').reset_index()
vSn['n'] = vSn['m']+'_'+vSn['k']
vSn = vSn.set_index(['s','n'])['vs']

Define types:

In [32]:
nmTypes = wd['vs'].index.levels[1].rename('n')

Adjusted data:

In [33]:
# pd.set_option('future.no_silent_downcasting', True)
# targets = CreateIOTargets(db)(vD0, vDn, vSn)
# adjustedData = AdjustIO(ws = db.ws)(targets.db('vD0'), targets.db('vD'), targets.db('vDn'), 
#                                     targets.db('vS0'), targets.db('vS'), targets.db('vSn'))

In [34]:
vS = db('vS').groupby('s').sum()

Output after adjustments:

In [36]:
vSdiff = vS.add( - vSn.groupby('s').sum(), fill_value=0)

In [38]:
# vDn.groupby('s').sum()